# 03 — End-to-End Agentic App

Runs the full LangGraph workflow on several sample tickets and shows:

1. Classification + routing decisions
2. RAG knowledge retrieval with confidence scoring
3. Tool invocation against the core and external databases
4. Resolution vs. escalation outcomes
5. Short-term (per-thread) and long-term (per-user) memory
6. The interactive `chat_interface()` REPL

> Requires `OPENAI_API_KEY` in `.env` (or environment).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agentic.logging_utils import configure_logging
from agentic.workflow import build_app
from agentic.runner import run_ticket
from utils import print_result, chat_interface

configure_logging()
app = build_app()
print(app.get_graph().draw_mermaid())

## 1. Scenario A — KB resolution (how-to)

In [ ]:
result = run_ticket(app, {
    'ticket_id': 'tkt_demo_kb',
    'user_id':   'usr_001',
    'subject':   'How do I turn on 2FA?',
    'body':      'I want to enable two-factor authentication on my Cult Pass account.',
    'channel':   'web',
    'urgency':   'normal',
})
print_result(result)

## 2. Scenario B — Tool-driven refund (uses cultpass_member_lookup + process_refund)

In [ ]:
result = run_ticket(app, {
    'ticket_id': 'tkt_demo_refund',
    'user_id':   'usr_002',
    'subject':   'Charged twice this month',
    'body':      'You billed me $9.99 twice for May membership. Please refund the duplicate.',
    'channel':   'email',
    'urgency':   'high',
})
print_result(result)

## 3. Scenario C — Cancel a CultPass booking via the external tool

In [ ]:
result = run_ticket(app, {
    'ticket_id': 'tkt_demo_cancel',
    'user_id':   'usr_006',
    'subject':   'Need to cancel my Hamlet booking',
    'body':      "Please cancel my booking cp_b_007 (Hamlet at Donmar Warehouse). Something came up.",
    'channel':   'chat',
    'urgency':   'normal',
})
print_result(result)

## 4. Scenario D — Escalation (no relevant article + critical urgency)

In [ ]:
result = run_ticket(app, {
    'ticket_id': 'tkt_demo_esc',
    'user_id':   'usr_003',
    'subject':   'Wallet QR not loading at venue right now',
    'body':      'I am at the venue and the QR will not load. Doors closed in 10 minutes!',
    'channel':   'chat',
    'urgency':   'critical',
})
print_result(result)

## 5. Memory in action — follow-up in the same thread

Reusing the thread_id from Scenario B continues that conversation; the
checkpointer makes the prior state available.

In [ ]:
result = run_ticket(app, {
    'ticket_id': 'tkt_demo_followup',
    'user_id':   'usr_002',
    'subject':   'Quick follow-up to my refund',
    'body':      'When should I see the refund on my statement?',
    'channel':   'email',
    'urgency':   'normal',
    'thread_id': 'thread-tkt_demo_refund',
})
print_result(result)

## 6. Inspect long-term memory and persisted ticket state

In [ ]:
from agentic.memory import get_long_term_store
for it in get_long_term_store().search(('customer','usr_002')):
    print(it.key, '->', it.value)

In [ ]:
from data.core import db
from tabulate import tabulate
print(tabulate(db.fetch_all('''
    SELECT t.ticket_id, t.status, m.category, m.urgency, m.sentiment, m.confidence, m.routed_to
      FROM Ticket t JOIN TicketMetadata m USING(ticket_id)
     WHERE t.ticket_id LIKE 'tkt_demo_%'
     ORDER BY t.ticket_id
'''), headers='keys'))

## 7. Optional — interactive chat shell

Uncomment to launch `chat_interface()`. It reuses the compiled `app` and
pins one `thread_id` per session, so short-term memory works.

In [ ]:
# chat_interface(app, default_user='usr_002')